# Codex API 调用调试手册

通过分析 `~/.codex/logs_2.sqlite` 日志数据库，查看 Codex 桌面 App 的后端 API 调用细节。

> 💡 基于 Responses API（OpenAI 新格式），按 **Shift + Enter** 逐步运行

In [ ]:
# 🔧 初始化 - 连接数据库
import sqlite3
import json
import re
from datetime import datetime

DB_PATH = "/Users/weike/.codex/logs_2.sqlite"
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

def parse_sse(body):
    # 从 SSE event: ... 日志中提取 JSON
    m = re.search(r'SSE event: (.+)', body, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1))
        except json.JSONDecodeError:
            return None
    return None

# 基本信息
cursor.execute("SELECT COUNT(*) FROM logs")
total = cursor.fetchone()[0]
print(f"✅ 数据库连接成功")
print(f"📊 总日志数: {total:,} 条")

cursor.execute("SELECT ts FROM logs ORDER BY id DESC LIMIT 1")
latest_ts = cursor.fetchone()[0]
print(f"⏰ 最新日志时间: {datetime.fromtimestamp(latest_ts)}")


## 1. 日志模块分布

了解日志结构，找到 API 相关的模块。

In [ ]:
# 📋 所有日志模块 Top 20
cursor.execute("""
    SELECT target, COUNT(*) as cnt
    FROM logs
    GROUP BY target
    ORDER BY cnt DESC
    LIMIT 20
""")
rows = cursor.fetchall()
print(f"共 {len(rows)} 个主要模块:\n")
for target, cnt in rows:
    print(f"  {target:60s} {cnt:>6,}")


## 2. SSE 响应事件类型

`codex_api::sse::responses` 模块记录了所有 API 响应事件。

In [ ]:
# 🌊 SSE 事件类型分布
cursor.execute("""
    SELECT feedback_log_body, id, ts
    FROM logs
    WHERE target = 'codex_api::sse::responses'
    ORDER BY id DESC
    LIMIT 300
""")
rows = cursor.fetchall()

event_types = {}
for body, log_id, ts in rows:
    data = parse_sse(body)
    if data:
        evt = data.get('type', 'unknown')
        event_types[evt] = event_types.get(evt, 0) + 1

print(f"📊 事件类型 (最近 {len(rows)} 条 SSE 日志):\n")
for evt, cnt in sorted(event_types.items(), key=lambda x: -x[1]):
    print(f"  {evt:45s} {cnt:>4} 次")


## 3. 最近一次完整请求 & 响应

从 `response.completed` 事件中提取完整的对话信息。

In [ ]:
# 💬 最近一次完整对话 (input + output)

# 先从 transport 模块提取输入消息 (请求体最完整)
cursor.execute("""
    SELECT id, ts, feedback_log_body
    FROM logs
    WHERE target = 'codex_http_client::transport'
    ORDER BY id DESC
    LIMIT 1
""")
row = cursor.fetchone()

input_messages = []
req_model = '?'
req_tools = 0

if row:
    log_id, ts, body = row
    # 提取 input 数组
    m = re.search(r'"input"\s*:\s*(\[.*?\])\s*,\s*"(?:model|tools)"', body, re.DOTALL)
    if m:
        try:
            input_messages = json.loads(m.group(1))
        except:
            pass
    # 提取模型名
    mm = re.search(r'"model"\s*:\s*"([^"]+)"', body)
    if mm:
        req_model = mm.group(1)
    # 统计工具数
    tm = re.search(r'"tools"\s*:\s*(\[)', body)
    if tm:
        # 数 name 出现次数
        tool_section = body[body.find('"tools"'):body.find('"tools"')+5000]
        req_tools = len(re.findall(r'"name"\s*:\s*"', tool_section))

# 再从 SSE response.completed 提取输出
cursor.execute("""
    SELECT id, ts, feedback_log_body
    FROM logs
    WHERE target = 'codex_api::sse::responses'
      AND feedback_log_body LIKE '%response.completed%'
    ORDER BY id DESC
    LIMIT 3
""")
rows = cursor.fetchall()

for log_id, ts, body in rows:
    data = parse_sse(body)
    if not data:
        continue
    resp = data.get('response', data)

    time_str = datetime.fromtimestamp(ts).strftime('%Y-%m-%d %H:%M:%S')
    print(f"📌 响应 ID: {resp.get("id", "N/A")}")
    print(f"   时间: {time_str}")
    print(f"   模型: {resp.get("model", req_model)}")
    print(f"   状态: {resp.get("status", "N/A")}")
    print(f"   工具数: {len(resp.get("tools", [])) or req_tools}")

    usage = resp.get('usage', {})
    if usage:
        inp_tok = usage.get('input_tokens', '?')
        out_tok = usage.get('output_tokens', '?')
        tot_tok = usage.get('total_tokens', '?')
        print(f"   用量: input={inp_tok}, output={out_tok}, total={tot_tok}")

    # 输入消息 (优先用 transport 里的，更全)
    if input_messages:
        print(f"\n📥 输入消息 ({len(input_messages)} 条，显示最近 6 条):\n")
        for i, msg in enumerate(input_messages[-6:], 1):
            role = msg.get('role', msg.get('type', 'unknown'))
            content = msg.get('content', '')
            text = ''
            if isinstance(content, list):
                for c in content:
                    if isinstance(c, dict):
                        if 'text' in c:
                            text += c['text']
                        elif c.get('type') in ('input_text', 'output_text'):
                            text += c.get('text', '')
            elif isinstance(content, str):
                text = content
            # function_call_output 类型
            if role == 'function_call_output':
                output_val = msg.get('output', '')
                if isinstance(output_val, str):
                    text = output_val[:200]
            text = str(text).strip()
            preview = text[:150] + '...' if len(text) > 150 else text
            print(f"  [{i}] {role}:")
            print(f"      {preview}")
            print()

    # 输出
    output = resp.get('output', [])
    print(f"📤 输出 ({len(output)} 项):\n")
    for idx_item, item in enumerate(output):
        itype = item.get('type', '?')
        status = item.get('status', '')
        role = item.get('role', '')
        label = itype + (f' ({role})' if role else '')
        print(f"  [{idx_item}] {label}  {status}")

        content = item.get('content', [])
        if isinstance(content, list):
            for c in content:
                if c.get('type') == 'output_text':
                    text = c.get('text', '')
                    preview = text[:300] + '...' if len(text) > 300 else text
                    print(f"      text: {preview}")
        if itype == 'function_call':
            print(f"      name: {item.get("name", "?")}")
            args = item.get('arguments', '')
            if isinstance(args, str):
                preview = args[:200] + '...' if len(args) > 200 else args
                print(f"      args: {preview}")
    print()
    break
else:
    print("未找到 response.completed 日志")


## 4. 工具 / Function Calling 分析

查看每次请求注册了哪些工具。

In [ ]:
# 🛠️  最近请求的工具列表
cursor.execute("""
    SELECT id, ts, feedback_log_body
    FROM logs
    WHERE target = 'codex_api::sse::responses'
      AND feedback_log_body LIKE '%response.completed%'
    ORDER BY id DESC
    LIMIT 3
""")
rows = cursor.fetchall()

for log_id, ts, body in rows:
    data = parse_sse(body)
    if not data:
        continue
    resp = data.get('response', data)
    tools = resp.get('tools', [])
    if not tools:
        continue

    time_str = datetime.fromtimestamp(ts).strftime('%H:%M:%S')
    print(f"📌 {time_str}  模型: {resp.get("model", "?")}")
    print(f"   已注册工具: {len(tools)} 个\n")

    for i, t in enumerate(tools, 1):
        name = t.get('name', 'unknown')
        ttype = t.get('type', '?')
        desc = t.get('description', '')[:80]
        print(f"  {i:2d}. {name}  ({ttype})")
        print(f"      {desc}")
    print()
    break
else:
    print("未找到带工具的请求")


## 5. 工具调用轨迹

查看一次响应中所有 function_call 的调用顺序和参数。

In [ ]:
# 🔄 一次响应中的完整工具调用轨迹
cursor.execute("""
    SELECT id, ts, feedback_log_body
    FROM logs
    WHERE target = 'codex_api::sse::responses'
      AND feedback_log_body LIKE '%response.completed%'
    ORDER BY id DESC
    LIMIT 5
""")
rows = cursor.fetchall()

for log_id, ts, body in rows:
    data = parse_sse(body)
    if not data:
        continue
    resp = data.get('response', data)
    output = resp.get('output', [])
    func_calls = [o for o in output if o.get('type') == 'function_call']
    if not func_calls:
        continue

    time_str = datetime.fromtimestamp(ts).strftime('%H:%M:%S')
    print(f"📌 {time_str}  模型: {resp.get("model", "?")}")
    print(f"   共 {len(func_calls)} 次工具调用:\n")

    for i, fc in enumerate(func_calls, 1):
        name = fc.get('name', '?')
        status = fc.get('status', '?')
        args = fc.get('arguments', '{}')
        try:
            if isinstance(args, str):
                args = json.loads(args)
        except:
            pass
        print(f"  [{i}] {name}  [{status}]")
        if isinstance(args, dict):
            for k, v in list(args.items())[:5]:
                v_str = str(v)[:60]
                print(f"      {k}: {v_str}")
            if len(args) > 5:
                print(f"      ... 等 {len(args)} 个参数")
        else:
            print(f"      args: {str(args)[:80]}")
        if fc.get('id'):
            print(f"      call_id: {fc["id"]}")
        print()
    break
else:
    print("未找到包含工具调用的响应")


## 6. 模型与 Token 用量统计

统计最近 N 次请求的模型和 token 消耗。

In [ ]:
# 📈 最近 20 次请求的模型 & 用量
cursor.execute("""
    SELECT id, ts, feedback_log_body
    FROM logs
    WHERE target = 'codex_api::sse::responses'
      AND feedback_log_body LIKE '%response.completed%'
    ORDER BY id DESC
    LIMIT 20
""")
rows = cursor.fetchall()

total_input = 0
total_output = 0
model_counts = {}
print(f"最近 {len(rows)} 次响应:\n")
header = f'  {"时间":<12s} {"模型":<35s} {"输入":>6s} {"输出":>6s} {"工具":>4s}'
print(header)
print(f"  {"-"*12} {"-"*35} {"-"*6} {"-"*6} {"-"*4}")

for log_id, ts, body in rows:
    data = parse_sse(body)
    if not data:
        continue
    resp = data.get('response', data)
    model = resp.get('model', '?')[:32]
    usage = resp.get('usage', {})
    in_tok = usage.get('input_tokens', 0)
    out_tok = usage.get('output_tokens', 0)
    tools_n = len(resp.get('tools', []))
    time_str = datetime.fromtimestamp(ts).strftime('%H:%M:%S')

    total_input += in_tok
    total_output += out_tok
    model_counts[model] = model_counts.get(model, 0) + 1

    print(f"  {time_str:<12s} {model:<35s} {in_tok:>6,} {out_tok:>6,} {tools_n:>4d}")

print(f"\n📊 合计: 输入 {total_input:,} tok, 输出 {total_output:,} tok, 总计 {total_input+total_output:,} tok")
print(f"🏷️  模型分布: {dict(model_counts)}")


## 7. HTTP 客户端请求日志

从 `codex_http_client::client` 模块查看底层 HTTP 请求细节。

In [ ]:
# 🌐 HTTP 客户端请求 (最近 10 条)
cursor.execute("""
    SELECT id, ts, level, feedback_log_body
    FROM logs
    WHERE target = 'codex_http_client::client'
    ORDER BY id DESC
    LIMIT 10
""")
rows = cursor.fetchall()
print(f"最近 {len(rows)} 条 HTTP client 日志:\n")
for log_id, ts, level, body in rows:
    time_str = datetime.fromtimestamp(ts).strftime('%H:%M:%S')
    print(f"【{log_id}】 {time_str}  [{level}]")
    print(f"   {body[:200]}")
    print()


## 8. 自定义关键词搜索

修改 `keyword` 搜索任意内容。

In [ ]:
# 🔎 自定义关键词搜索
keyword = "doubao"  # 👈 修改关键词

cursor.execute("""
    SELECT id, ts, target, level, feedback_log_body
    FROM logs
    WHERE feedback_log_body LIKE ?
    ORDER BY id DESC
    LIMIT 15
""", (f"%{keyword}%",))
rows = cursor.fetchall()
print(f"找到 {len(rows)} 条匹配日志:\n")
for log_id, ts, target, level, body in rows:
    time_str = datetime.fromtimestamp(ts).strftime('%H:%M:%S')
    print(f"【{log_id}】 {time_str}  [{level}] {target}")
    print(f"   {body[:180]}")
    print()


## 9. 错误 / 异常日志

查看 WARN 和 ERROR 级别的日志。

In [ ]:
# ⚠️ 错误日志 (最近 20 条)
cursor.execute("""
    SELECT id, ts, target, level, feedback_log_body
    FROM logs
    WHERE level IN ('ERROR', 'WARN')
    ORDER BY id DESC
    LIMIT 20
""")
rows = cursor.fetchall()
print(f"找到 {len(rows)} 条 WARN/ERROR 日志:\n")
for log_id, ts, target, level, body in rows:
    time_str = datetime.fromtimestamp(ts).strftime('%H:%M:%S')
    print(f"【{log_id}】 {time_str}  [{level}] {target}")
    print(f"   {body[:200]}")
    print()


In [ ]:
# 🔌 关闭连接
conn.close()
print("✅ 数据库连接已关闭")
